# שרת תמלול עברית — ivrit.ai על GPU של Colab

מריץ את מודל העברית של [ivrit.ai](https://huggingface.co/ivrit-ai) (`whisper-large-v3-turbo`) על GPU חינמי של Google Colab, וחושף אותו לאפליקציה דרך מנהרה של Cloudflare.

**לפני כל דיון:**
1. בתפריט: **Runtime → Change runtime type → T4 GPU** (פעם אחת).
2. **Runtime → Run all** (או Ctrl+F9).
3. בסוף יודפסו **כתובת** ו**קוד גישה** — הדבק אותם בעמוד "סיכום חי" באפליקציה, תחת "ivrit.ai".
4. השאר את הלשונית פתוחה בזמן הדיון. הכתובת משתנה בכל הרצה.

הסשן החינמי נסגר אחרי זמן חוסר פעילות — אם התמלול מפסיק, הרץ שוב.

In [ ]:
# 1) התקנות (כדקה)
!pip -q install faster-whisper==1.1.1 fastapi==0.115.6 uvicorn==0.34.0
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import torch
assert torch.cuda.is_available(), "אין GPU — בחר Runtime → Change runtime type → T4 GPU והרץ שוב"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) טעינת מודל העברית של ivrit.ai (בפעם הראשונה ~1.5GB הורדה)
from faster_whisper import WhisperModel

MODEL_ID = "ivrit-ai/whisper-large-v3-turbo-ct2"
model = WhisperModel(MODEL_ID, device="cuda", compute_type="float16")
print("המודל נטען:", MODEL_ID)

In [ ]:
# 3) שרת התמלול
import base64, io, secrets, threading
from typing import Optional

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

TOKEN = secrets.token_urlsafe(12)  # קוד גישה חדש בכל הרצה
MAX_AUDIO_B64 = 8_000_000           # ~4 דקות WAV 16kHz

app = FastAPI()
# The app calls this from the browser; the access code (not the origin) protects it.
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
gpu_lock = threading.Lock()


class TranscribeRequest(BaseModel):
    audio: str                    # base64 WAV
    prompt: Optional[str] = None  # names / terms / previous text, helps spelling


def check_token(authorization: Optional[str]) -> None:
    if authorization != f"Bearer {TOKEN}":
        raise HTTPException(status_code=401, detail="קוד גישה שגוי")


@app.get("/health")
def health(authorization: Optional[str] = Header(default=None)):
    check_token(authorization)
    return {"ok": True, "model": MODEL_ID}


@app.post("/transcribe")
def transcribe(req: TranscribeRequest, authorization: Optional[str] = Header(default=None)):
    check_token(authorization)
    if len(req.audio) > MAX_AUDIO_B64:
        raise HTTPException(status_code=413, detail="קטע אודיו גדול מדי")
    try:
        data = base64.b64decode(req.audio, validate=True)
    except Exception:
        raise HTTPException(status_code=400, detail="אודיו לא תקין")
    prompt = (req.prompt or "").strip()[:400] or None
    with gpu_lock:  # one transcription at a time on the GPU
        segments, info = model.transcribe(
            io.BytesIO(data),
            language="he",
            beam_size=5,
            vad_filter=True,                  # skip silence -> no invented text
            condition_on_previous_text=False,
            initial_prompt=prompt,
        )
        text = " ".join(s.text.strip() for s in segments).strip()
    return {"text": text, "duration": round(info.duration, 2)}


server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
print("השרת רץ מקומית על פורט 8000")

In [ ]:
# 4) מנהרה ציבורית (Cloudflare, בלי חשבון) + הדפסת הפרטים לאפליקציה
import re, subprocess, threading, time, urllib.request

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline and PUBLIC_URL is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
# Keep reading the tunnel's log so its output pipe never fills up and blocks it.
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
assert PUBLIC_URL, "המנהרה לא עלתה — הרץ את התא הזה שוב"

# Wait until the public URL actually answers.
for _ in range(30):
    try:
        req = urllib.request.Request(PUBLIC_URL + "/health", headers={"Authorization": f"Bearer {TOKEN}"})
        urllib.request.urlopen(req, timeout=10).read()
        break
    except Exception:
        time.sleep(2)

print("=" * 60)
print("הדבק באפליקציה (סיכום חי → ivrit.ai):")
print()
print("כתובת:    ", PUBLIC_URL)
print("קוד גישה: ", TOKEN)
print("=" * 60)

In [ ]:
# 5) השאר את התא הזה רץ בזמן הדיון — הוא מראה שהשרת חי ועוזר לסשן לא להירדם.
import time
while True:
    alive = tunnel.poll() is None
    print(time.strftime("%H:%M:%S"), "שרת פעיל" if alive else "המנהרה נפלה — הרץ את תא 4 שוב", flush=True)
    if not alive:
        break
    time.sleep(300)